It validate data,are they ok or not

for this pydantic also give some built in datatype using those we can also do this

In [12]:
from pydantic import BaseModel,EmailStr,AnyUrl
from typing import List,Dict,Optional


class Patient(BaseModel):
    name : str
    age : int
    email: EmailStr                # instead of giving str ,Emailstr restrict user to give proper mail format
    linkdin:AnyUrl                 # make sure that user is giving right format of url
    weight : float
    married : bool=False
    things : Optional[list]= None
    allergies : List[str]
    contact : Dict[str,str]

patient_info = {'name':"Sabbir","age":24,"weight":70.9,"allergies":["Hello","world"],
                'contact':{"mail":"abc@gmail","call":"1627113136"},"married":True,"email":"abc@gmail.com","linkdin":"https://linkdin.com/123"}

patient2 = Patient(**patient_info)

def insertion(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.things)
    print(patient.email)
    print(patient.linkdin)


insertion(patient2)

Sabbir
24
None
abc@gmail.com
https://linkdin.com/123


Now if we wanna apply custom data range(Field)

In [21]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field
from typing import List,Dict,Optional


class Patient(BaseModel):
    name : str = Field(max_length=15)
    age : int = Field(ge=18,lt=40)
    email: EmailStr                # instead of giving str ,Emailstr restrict user to give proper mail format
    linkdin:AnyUrl                 # make sure that user is giving right format of url
    weight : float = Field(gt=45)
    married : bool=False
    things : Optional[list]= None
    allergies : List[str] = Field(max_length=2)
    contact : Dict[str,str]

patient_info = {'name':"MR Sabbir","age":19,"weight":75.9,"allergies":["Hello","world"],
                'contact':{"mail":"abc@gmail","call":"1627113136"},"married":True,"email":"abc@gmail.com","linkdin":"https://linkdin.com/123"}

patient2 = Patient(**patient_info)

def insertion(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.things)
    print(patient.email)
    print(patient.linkdin)


insertion(patient2)

MR Sabbir
19
None
abc@gmail.com
https://linkdin.com/123


This field function is not only used to do these type of data validation but also used for adding meta data, about the pydantic class

for that work we have to combine Annotated and Field these two function

Annotated[datatype,Field(constraint, title= , description= ,examples= )]

Type coercion: it is like if I give "24" in an int space ,pydantic is smart enough to understand and let it as a int,  but it is alwyas not good and we always don't want this, so we can strict this thing

In [27]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field
from typing import List,Dict,Optional,Annotated


class Patient(BaseModel):
    name : str = Annotated[str, Field(max_length=20, title="Name of the Patient",description="Give me the name of the patient in 20 char"
                                      ,examples=['sabbir','sazzad'])]
    age : int = Field(ge=18,lt=40)
    email: EmailStr                # instead of giving str ,Emailstr restrict user to give proper mail format
    linkdin:AnyUrl                 # make sure that user is giving right format of url
    weight : Annotated[float, Field(gt=45,strict=True)]
    married : Annotated[bool,Field(default=None,description="Is the patient married or unmarried")]
    things : Optional[list]= None
    allergies : Annotated[List[str],Field(default=None, max_length=5)]
    contact : Dict[str,str]

patient_info = {'name':"MR Sabbir","age":19,"weight":75.9,"allergies":["Hello","world"],
                'contact':{"mail":"abc@gmail","call":"1627113136"},"married":True,"email":"abc@gmail.com","linkdin":"https://linkdin.com/123"}

patient2 = Patient(**patient_info)

def insertion(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.things)
    print(patient.email)
    print(patient.linkdin)


insertion(patient2)

MR Sabbir
19
None
abc@gmail.com
https://linkdin.com/123


These are the default things common for any purpose, but if we wanna make something unique for our business,like whoever have uiu mail will get this service otherwise not,so have to check uiu mail or not

It also can do data transformation like ,transforming the mail into smaller, making name in capital letter anyhow user uploads

### Field Validator
used for one single data validation

In [ ]:
from pydantic import BaseModel,EmailStr,Field,field_validator
from typing import List,Dict,Optional,Annotated


class Patient(BaseModel):
    name : str
    age : int
    email: EmailStr
    weight : float
    married : bool 
    allergies : List[str]
    contact : Dict[str,str]


    @field_validator('email')
    @classmethod
    def email_validator(cls,value):
        valid_domains = ["uiu.ac.bd",'nsu.ac.bd']
        domain_name = value.split("@")[-1]
        if domain_name.lower() not in valid_domains:
            raise ValueError("Not a valid domain")
        else :
            return value.lower()

    @field_validator("name")
    @classmethod
    def name_validator(cls,value):
        return value.upper()

patient_info = {'name':"Sabbir","age":24,"weight":70.9,"allergies":["Hello","world"],
                'contact':{"mail":"abc@gmail","call":"1627113136"},"married":True,'email':"abc@uiu.ac.bd"}

patient2 = Patient(**patient_info)

def insertion(patient:Patient):
    print(patient.name)
    print(patient.age)

insertion(patient2) 

SABBIR
24


we can set mode in field validator that is by default "after", 

if asks do I need to work after type coercion or before coercion?

In [47]:
from pydantic import BaseModel,Field,field_validator,EmailStr
from typing import List,Dict,Optional,Annotated


class Patient(BaseModel):
    name : str
    age : int
    email: EmailStr
    weight : float
    married : bool 
    allergies : List[str]
    contact : Dict[str,str]


    @field_validator('email')
    @classmethod
    def email_validator(cls,value):
        valid_domains = ["uiu.ac.bd",'nsu.ac.bd']
        domain_name = value.split("@")[-1]
        if domain_name.lower() not in valid_domains:
            raise ValueError("Not a valid domain")
        else :
            return value.lower()

    @field_validator("name")
    @classmethod
    def name_validator(cls,value):
        return value.upper()


    @field_validator("age",mode="before")
    @classmethod
    def age_validator(cls,value):
        if 0<value<100:
            return value
        else:
            ValueError("Value should be in between 0 and 100")
patient_info = {'name':"Sabbir","age":24,"weight":70.9,"allergies":["Hello","world"],
                'contact':{"mail":"abc@gmail","call":"1627113136"},"married":True,'email':"abc@uiu.ac.bd"}

patient2 = Patient(**patient_info)

def insertion(patient:Patient):
    print(patient.name)
    print(patient.age)

insertion(patient2) 

SABBIR
24


but if we keep it default (after) it will work

### So field_validator have two modes one is before and another one is after